# 공공데이터 활용 항만 물류 데이터 수집 실습 (port_data_lab)

공공데이터포털(data.go.kr)의 항만·해운 데이터를 Python으로 **수집 → 정제 → 적재 → 분석**하는 강의용 노트북입니다.
인증키가 없어도 **MOCK 모드(가상 데이터)** 로 모든 셀이 끝까지 실행됩니다.

> ⚠️ `sample_data/` 의 데이터는 실습용 **가상 데이터**이며 실제 항만 통계가 아닙니다.

### 구성
| 섹션 | 내용 | 원본 파일 |
|---|---|---|
| 0 | 환경 준비 | `requirements.txt` |
| 1 | 가상 샘플 데이터 생성 | `sample_data/make_sample_data.py` |
| 2 | 공용 수집 클라이언트 (재시도, 페이지 순회, 오류코드 해석, MOCK) | `portdata/client.py` |
| 3 | [예제 1] 파일데이터 자동변환 API — JSON 수집 | `ex01_odcloud_json.py` |
| 4 | [예제 2] 기관 REST API — XML 수집 + 자료형 변환 | `ex02_rest_xml.py` |
| 5 | [예제 3] 파일데이터 CSV(CP949) 읽기 | `ex03_file_csv.py` |
| 6 | [예제 4] 수집 파이프라인 → SQLite 적재 (UPSERT, 수집 로그) | `ex04_pipeline.py` |
| 7 | [예제 5] 분석·시각화 | `ex05_analysis.py` |

### 두 가지 API 형태 비교
| 구분 | 파일데이터 자동변환 API | 기관 REST API |
|---|---|---|
| 호스트 | `api.odcloud.kr` | `apis.data.go.kr` |
| 페이지 변수 | `page`, `perPage` | `pageNo`, `numOfRows` |
| 응답 | JSON (`data`, `totalCount`) | 주로 XML (`body/items/item`) |
| 성격 | 파일(CSV)을 API로 변환, 갱신 주기 김 | 기관 시스템 직접 연계, 실시간성 |

### 실제 데이터로 전환하기
1. data.go.kr 회원가입 → 데이터 검색(예: `부산항만공사`, `인천항만공사`, `항만운영`, `컨테이너`)
2. 데이터 상세에서 **[오픈API] 탭 → 활용신청** (개발계정은 대부분 자동승인)
3. 마이페이지 → 인증키 **Decoding** 키 복사 → `.env.example` 을 `.env` 로 복사해 붙여넣기
4. 각 예제의 `URL`, `EXTRA` 를 포털 **상세기능/Swagger** 의 요청주소·요청변수로 교체
5. 응답 항목명이 다르면 예제 4의 `clean_*` 함수에서 컬럼 매핑만 수정

## 0. 환경 준비

필요 패키지: `requests`, `pandas`, `matplotlib`. 설치되어 있지 않다면 아래 셀의 주석을 해제해 실행하세요.

노트북은 이 파일이 있는 폴더(`port_data_lab/`)를 작업 디렉터리로 사용합니다.
결과물은 `output/` 폴더에 저장됩니다.

In [ ]:
# %pip install "requests>=2.31" "pandas>=2.0" "matplotlib>=3.7"

In [ ]:
from __future__ import annotations

import csv
import json
import os
import random
import sqlite3
import time
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from pathlib import Path
from typing import Iterator

import matplotlib.pyplot as plt
import pandas as pd
import requests
from matplotlib import font_manager

%matplotlib inline

BASE_DIR = Path.cwd()                 # 노트북이 있는 폴더 (port_data_lab/)
SAMPLE_DIR = BASE_DIR / "sample_data"
OUT = BASE_DIR / "output"
SAMPLE_DIR.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
print("작업 폴더:", BASE_DIR)

## 1. 가상 샘플 데이터 생성 (최초 1회)

강의 실습용 **가상(synthetic) 데이터** 생성기입니다.
※ 실제 통계가 아닙니다. 인증키 없이 코드 흐름을 실습하기 위한 용도이며,
컬럼 구조만 공공데이터포털 항만 데이터와 유사하게 만들었습니다.

- `busan_container_monthly.json` : 월별 컨테이너 처리량 (odcloud JSON 형태) — 부두별, 수출입별
- `vessel_arrivals.json` : 선박 입출항 내역 (REST XML 형태로 변환되어 제공)
- `yearly_teu_cp949.csv` : 파일데이터(CSV) — 포털에서 내려받는 파일처럼 CP949 인코딩

In [ ]:
random.seed(42)

# 1) 월별 컨테이너 처리량 (odcloud JSON 형태) — 부두별, 수출입별
terminals = ["신항1부두", "신항2부두", "신항3부두", "신항4부두", "북항신선대", "북항감만"]
base = {"신항1부두": 95000, "신항2부두": 120000, "신항3부두": 80000,
        "신항4부두": 70000, "북항신선대": 60000, "북항감만": 40000}
rows = []
for year in range(2021, 2026):
    for month in range(1, 13):
        season = 1 + 0.06 * (month in (3, 4, 10, 11)) - 0.05 * (month in (1, 2))
        growth = 1 + 0.025 * (year - 2021)
        for t in terminals:
            for io in ("수출", "수입"):
                teu = base[t] / 2 * season * growth * random.uniform(0.92, 1.08)
                if t.startswith("북항"):
                    teu *= 1 - 0.04 * (year - 2021)  # 신항 이전 효과(가상)
                rows.append({"연도": year, "월": month, "부두명": t, "수출입구분": io,
                             "적컨TEU": round(teu * 0.82), "공컨TEU": round(teu * 0.18)})
(SAMPLE_DIR / "busan_container_monthly.json").write_text(
    json.dumps(rows, ensure_ascii=False, indent=1), encoding="utf-8")

# 2) 선박 입출항 내역 (REST XML 형태로 변환되어 제공)
ship_types = ["컨테이너선", "컨테이너선", "컨테이너선", "일반화물선", "유조선", "자동차운반선"]
flags = ["KR", "PA", "LR", "MH", "SG", "HK", "CN"]
vessels = []
start = datetime(2026, 9, 1)
for i in range(240):
    eta = start + timedelta(hours=random.randint(0, 24 * 30))
    stay = random.uniform(8, 60)
    vessels.append({
        "callSign": f"D{random.randint(1000, 9999)}{chr(65 + i % 26)}",
        "vsslNm": f"SAMPLE VESSEL {i + 1:03d}",
        "vsslKind": random.choice(ship_types),
        "flag": random.choice(flags),
        "grtg": random.choice([9000, 28000, 55000, 98000, 150000]) + random.randint(-2000, 2000),
        "berthNm": random.choice(terminals),
        "etryDt": eta.strftime("%Y%m%d%H%M"),
        "tkoffDt": (eta + timedelta(hours=stay)).strftime("%Y%m%d%H%M"),
        "prvsPort": random.choice(["SHANGHAI", "NINGBO", "TOKYO", "SINGAPORE", "LOS ANGELES", "QINGDAO"]),
    })
(SAMPLE_DIR / "vessel_arrivals.json").write_text(
    json.dumps(vessels, ensure_ascii=False, indent=1), encoding="utf-8")

# 3) 파일데이터(CSV) — 포털에서 내려받는 파일은 대부분 CP949(EUC-KR) 인코딩
with open(SAMPLE_DIR / "yearly_teu_cp949.csv", "w", encoding="cp949", newline="") as f:
    w = csv.writer(f)
    w.writerow(["연도", "항만", "수출(TEU)", "수입(TEU)", "환적(TEU)"])
    for year in range(2019, 2026):
        for port, b in [("부산항", 2200), ("인천항", 330), ("광양항", 210), ("울산항", 45)]:
            g = 1 + 0.03 * (year - 2019) + random.uniform(-0.04, 0.04)
            tr = 0.55 if port == "부산항" else 0.05
            total = b * 10000 * g
            w.writerow([year, port, f"{total*(1-tr)/2:,.0f}", f"{total*(1-tr)/2:,.0f}",
                        f"{total*tr:,.0f}"])

print("가상 샘플 데이터 생성 완료:", sorted(p.name for p in SAMPLE_DIR.glob("*") if p.suffix != ".py"))

## 2. 공용 수집 클라이언트 (`portdata/client.py`)

공공데이터포털의 오픈 API는 크게 두 가지 형태로 제공됩니다.

1. **파일데이터 자동변환 API** (`api.odcloud.kr`)
   - URL 형태 : `https://api.odcloud.kr/api/{데이터ID}/v1/uddi:{UUID}`
   - 페이지 파라미터 : `page`, `perPage`
   - 응답 : JSON `{"page", "perPage", "totalCount", "currentCount", "matchCount", "data": [...]}`
2. **기관 제공 REST API** (`apis.data.go.kr`)
   - URL 형태 : `https://apis.data.go.kr/{기관코드}/{서비스명}/{오퍼레이션}`
   - 페이지 파라미터 : `pageNo`, `numOfRows`
   - 응답 : 주로 XML `<response><header><resultCode/>...</header><body><items><item/>...</items><numOfRows/><pageNo/><totalCount/></body></response>`

정확한 요청주소(uddi 경로, 오퍼레이션명)와 요청변수는 데이터마다 다르므로
반드시 포털의 **[활용신청 > 상세기능/Swagger]** 화면에서 복사해 사용합니다.

**MOCK 모드**: 인증키가 없거나 `PORTDATA_MOCK=1` 이면 `sample_data/` 의 가상 데이터로 동일한 코드를 실행할 수 있습니다.

### 2-1. 오류코드와 인증키 로딩

In [ ]:
# data.go.kr 공통 오류코드 (게이트웨이 응답) — 포털 명세서의 '에러코드' 표 기준
ERROR_CODES = {
    "00": "정상",
    "01": "어플리케이션 에러",
    "03": "데이터 없음(NODATA_ERROR)",
    "04": "HTTP 에러",
    "05": "서비스 연결 실패 / 응답 시간 초과",
    "10": "잘못된 요청 파라미터",
    "11": "필수 요청 파라미터 누락",
    "12": "해당 오픈API 서비스가 없거나 폐기됨",
    "20": "서비스 접근 거부(활용신청 미승인 등)",
    "22": "일일 호출 허용량 초과",
    "30": "등록되지 않은 서비스키",
    "31": "서비스키 사용 기한 만료",
    "32": "등록되지 않은 IP",
    "99": "기타 에러",
}

# 재시도해도 소용없는 오류(키·권한·파라미터 문제) — 즉시 중단
NON_RETRYABLE = {"10", "11", "12", "20", "22", "30", "31", "32"}


class DataGoKrError(RuntimeError):
    def __init__(self, code: str, message: str):
        self.code = code
        super().__init__(f"[{code}] {ERROR_CODES.get(code, '알 수 없는 오류')} - {message}")


def _load_dotenv() -> None:
    """작업 폴더의 .env 를 읽어 환경변수에 없을 때만 채운다(python-dotenv 없이)."""
    env_file = BASE_DIR / ".env"
    if not env_file.exists():
        return
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))


def load_service_key() -> str | None:
    """환경변수 → .env 파일 순서로 인증키를 찾는다. 코드에 키를 직접 쓰지 않는다."""
    _load_dotenv()
    key = os.getenv("DATA_GO_KR_KEY", "").strip()
    return key or None

### 2-2. `DataGoKrClient` 클래스
- `_get` : 재시도(지수 백오프)가 포함된 GET 요청
- `fetch_odcloud_page` / `iter_odcloud` : JSON API 한 페이지 / 전체 페이지 순회
- `parse_xml` / `fetch_xml_page` / `iter_xml` : XML 응답 파싱(오류 판별 포함) / 한 페이지 / 전체 순회
- `_mock_odcloud` / `_mock_xml` : 가상 데이터로 실제 응답 형태를 흉내

In [ ]:
class DataGoKrClient:
    def __init__(self, service_key: str | None = None, mock: bool | None = None,
                 timeout: int = 15, max_retries: int = 3, sleep_between: float = 0.2):
        self.service_key = service_key or load_service_key()
        env_mock = os.getenv("PORTDATA_MOCK") == "1"
        self.mock = mock if mock is not None else (env_mock or not self.service_key)
        self.timeout = timeout
        self.max_retries = max_retries
        self.sleep_between = sleep_between  # 서버 부하·호출 제한을 고려한 페이지 간 대기
        self.session = requests.Session()

    # ------------------------------------------------------------------ 공통
    def _get(self, url: str, params: dict) -> requests.Response:
        """재시도(지수 백오프)가 포함된 GET 요청."""
        # 인증키는 requests가 다시 인코딩하지 않도록 '디코딩(Decoding) 키'를 사용한다.
        params = {"serviceKey": self.service_key, **params}
        last_exc: Exception | None = None
        for attempt in range(1, self.max_retries + 1):
            try:
                resp = self.session.get(url, params=params, timeout=self.timeout)
                if resp.status_code in (429, 500, 502, 503, 504):
                    raise requests.HTTPError(f"HTTP {resp.status_code}", response=resp)
                resp.raise_for_status()
                return resp
            except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as exc:
                last_exc = exc
                wait = 2 ** (attempt - 1)
                print(f"  ! 요청 실패({attempt}/{self.max_retries}): {exc} → {wait}s 후 재시도")
                time.sleep(wait)
        raise RuntimeError(f"요청 최종 실패: {url}") from last_exc

    # ----------------------------------------------- 1) odcloud JSON (파일데이터 API)
    def fetch_odcloud_page(self, url: str, page: int = 1, per_page: int = 100,
                           extra: dict | None = None) -> dict:
        if self.mock:
            return self._mock_odcloud(url, page, per_page)
        resp = self._get(url, {"page": page, "perPage": per_page, **(extra or {})})
        payload = resp.json()
        if "data" not in payload:  # 오류 시 {"code": -4, "msg": "..."} 형태로 옴
            raise DataGoKrError(str(payload.get("code", "99")), str(payload.get("msg", payload)))
        return payload

    def iter_odcloud(self, url: str, per_page: int = 100, extra: dict | None = None) -> Iterator[dict]:
        """totalCount를 보고 마지막 페이지까지 자동으로 순회하며 레코드를 하나씩 반환."""
        page = 1
        while True:
            payload = self.fetch_odcloud_page(url, page, per_page, extra)
            rows = payload.get("data", [])
            yield from rows
            total = int(payload.get("totalCount", 0))
            print(f"  · page {page}: {len(rows)}건 (누적 {min(page * per_page, total)}/{total})")
            if not rows or page * per_page >= total:
                break
            page += 1
            time.sleep(self.sleep_between)

    # ----------------------------------------------- 2) REST XML (기관 API)
    @staticmethod
    def parse_xml(text: str) -> tuple[dict, list[dict]]:
        """XML 응답 → (메타정보, item 리스트). 오류 응답도 여기서 판별한다."""
        root = ET.fromstring(text)
        # 게이트웨이 오류: <OpenAPI_ServiceResponse><cmmMsgHeader>...<returnReasonCode>30</...>
        reason = root.findtext(".//returnReasonCode")
        if reason:
            raise DataGoKrError(reason, root.findtext(".//returnAuthMsg") or root.findtext(".//errMsg") or "")
        code = (root.findtext(".//header/resultCode") or "00").strip()
        msg = root.findtext(".//header/resultMsg") or ""
        if code not in ("00", "0", "0000", "INFO-000"):
            if code == "03":
                return {"totalCount": 0}, []
            raise DataGoKrError(code, msg)
        body = root.find(".//body")
        meta = {
            "totalCount": int(body.findtext("totalCount") or 0) if body is not None else 0,
            "pageNo": int(body.findtext("pageNo") or 1) if body is not None else 1,
            "numOfRows": int(body.findtext("numOfRows") or 0) if body is not None else 0,
        }
        items = [{child.tag: (child.text or "").strip() for child in item}
                 for item in root.iter("item")]
        return meta, items

    def fetch_xml_page(self, url: str, page_no: int = 1, num_rows: int = 100,
                       extra: dict | None = None) -> tuple[dict, list[dict]]:
        if self.mock:
            return self.parse_xml(self._mock_xml(url, page_no, num_rows))
        resp = self._get(url, {"pageNo": page_no, "numOfRows": num_rows, **(extra or {})})
        resp.encoding = resp.encoding or "utf-8"
        return self.parse_xml(resp.text)

    def iter_xml(self, url: str, num_rows: int = 100, extra: dict | None = None) -> Iterator[dict]:
        page = 1
        while True:
            meta, items = self.fetch_xml_page(url, page, num_rows, extra)
            yield from items
            total = meta.get("totalCount", 0)
            print(f"  · pageNo {page}: {len(items)}건 (전체 {total})")
            if not items or page * num_rows >= total:
                break
            page += 1
            time.sleep(self.sleep_between)

    # ----------------------------------------------- MOCK (강의 실습용 가상 데이터)
    def _mock_odcloud(self, url: str, page: int, per_page: int) -> dict:
        rows = json.loads((SAMPLE_DIR / "busan_container_monthly.json").read_text(encoding="utf-8"))
        start = (page - 1) * per_page
        chunk = rows[start:start + per_page]
        return {"page": page, "perPage": per_page, "totalCount": len(rows),
                "currentCount": len(chunk), "matchCount": len(rows), "data": chunk}

    def _mock_xml(self, url: str, page_no: int, num_rows: int) -> str:
        rows = json.loads((SAMPLE_DIR / "vessel_arrivals.json").read_text(encoding="utf-8"))
        start = (page_no - 1) * num_rows
        chunk = rows[start:start + num_rows]
        items = "".join(
            "<item>" + "".join(f"<{k}>{v}</{k}>" for k, v in r.items()) + "</item>" for r in chunk
        )
        return (
            '<?xml version="1.0" encoding="UTF-8"?><response>'
            "<header><resultCode>00</resultCode><resultMsg>NORMAL SERVICE.</resultMsg></header>"
            f"<body><items>{items}</items><numOfRows>{num_rows}</numOfRows>"
            f"<pageNo>{page_no}</pageNo><totalCount>{len(rows)}</totalCount></body></response>"
        )

In [ ]:
client = DataGoKrClient()
print(f"MOCK 모드: {client.mock}")

## 3. [예제 1] 파일데이터 자동변환 API (`api.odcloud.kr`) — JSON 수집

대상 예시 : 부산항만공사_외내항컨테이너통합집계정보 / 부산항 컨테이너 수송통계 등 (포털에서 '오픈API' 탭이 있는 파일데이터)

준비
1. data.go.kr 로그인 → 데이터 검색 → [오픈API] 탭 → 활용신청 (개발계정 자동승인)
2. 마이페이지 > 데이터활용 > 인증키(Decoding) 복사 → `.env` 에 `DATA_GO_KR_KEY=...`
3. 상세기능의 요청주소(`uddi:...`)를 아래 `URL`에 붙여넣기 (키가 없으면 가상 데이터 MOCK 모드)

In [ ]:
# ▼ 포털 '상세기능'에서 복사한 요청주소로 교체하세요 (아래는 형식 예시)
URL = "https://api.odcloud.kr/api/15147941/v1/uddi:REPLACE-WITH-YOUR-UDDI"

# STEP 1. 한 페이지만 호출해서 응답 구조부터 확인하는 습관
first = client.fetch_odcloud_page(URL, page=1, per_page=5)
print("응답 키:", list(first.keys()))
print("전체 건수:", first["totalCount"])
print("첫 레코드:", first["data"][0])

In [ ]:
# STEP 2. 전체 페이지 순회 수집
records = list(client.iter_odcloud(URL, per_page=200))
df = pd.DataFrame(records)
df.head()

In [ ]:
# STEP 3. 기본 점검: 건수, 결측, 자료형
print(f"수집 완료: {len(df):,}행 × {df.shape[1]}열")
print(df.dtypes)
print("결측치:\n", df.isna().sum())

In [ ]:
# STEP 4. 저장 — 엑셀에서 한글이 깨지지 않도록 utf-8-sig
path = OUT / "container_monthly_raw.csv"
df.to_csv(path, index=False, encoding="utf-8-sig")
print("저장:", path)

## 4. [예제 2] 기관 제공 REST API (`apis.data.go.kr`) — XML 수집

대상 예시 : 인천항만공사_항만운영 통계정보 조회 서비스, 인천항 선석 이용 정보 조회 서비스,
해양수산부 항만·선박 관련 조회 서비스 등 (응답 포맷 XML)

핵심 포인트
- 페이지 파라미터 이름이 `pageNo` / `numOfRows` 로 odcloud(`page`/`perPage`)와 다르다.
- HTTP 200 이어도 본문에 오류코드(`resultCode`, `returnReasonCode`)가 담겨 올 수 있다.
- 응답 항목명(`vsslNm`, `etryDt` 등)은 서비스마다 다르므로 명세서의 '응답메시지' 표를 확인.

In [ ]:
# ▼ 포털 Swagger / 상세기능의 '요청주소'로 교체 (형식 예시)
URL = "https://apis.data.go.kr/B551xxx/REPLACE_SERVICE/REPLACE_OPERATION"

# ▼ 서비스별 추가 요청변수 (예: 조회기간, 항코드). 명세서의 '요청변수' 표 참고
EXTRA = {
    # "sde": "20260901", "ede": "20260930", "prtAgCd": "020",
}

try:
    records = list(client.iter_xml(URL, num_rows=100, extra=EXTRA))
except DataGoKrError as e:
    # 30: 키 미등록 → 키 복사 오류 또는 승인 반영 대기(발급 후 1~2시간 소요될 수 있음)
    # 20: 접근 거부 → 해당 서비스 활용신청 여부 확인
    # 22: 일일 트래픽 초과 → 개발계정은 보통 일 10,000건
    print("API 오류:", e)
    records = []

df = pd.DataFrame(records)
print(f"수집 완료: {len(df):,}건")

In [ ]:
# XML 값은 모두 문자열 → 자료형 변환이 필요
for col in ("etryDt", "tkoffDt"):
    if col in df:
        df[col] = pd.to_datetime(df[col], format="%Y%m%d%H%M", errors="coerce")
if "grtg" in df:
    df["grtg"] = pd.to_numeric(df["grtg"], errors="coerce")

# 파생변수: 재항시간(시간)
if {"etryDt", "tkoffDt"} <= set(df.columns):
    df["재항시간_h"] = (df["tkoffDt"] - df["etryDt"]).dt.total_seconds() / 3600

df.to_csv(OUT / "vessel_arrivals.csv", index=False, encoding="utf-8-sig")
print("저장:", OUT / "vessel_arrivals.csv")
df.head()

## 5. [예제 3] 파일데이터(CSV) 직접 읽기 — API 신청 없이 바로 쓰는 방법

포털의 파일데이터는 로그인 없이 다운로드할 수 있는 경우가 많다.
연 단위 통계처럼 갱신이 드문 데이터는 API보다 파일이 간편하다.

**주의** : 포털 CSV는 대부분 CP949(EUC-KR) 인코딩, 숫자에 천 단위 쉼표가 들어 있음.

In [ ]:
SRC = SAMPLE_DIR / "yearly_teu_cp949.csv"  # 내려받은 파일 경로로 교체


def read_korean_csv(path: Path) -> pd.DataFrame:
    """인코딩을 순서대로 시도해 읽는다."""
    for enc in ("utf-8-sig", "cp949", "euc-kr"):
        try:
            df = pd.read_csv(path, encoding=enc, thousands=",")
            print(f"인코딩 '{enc}' 로 읽기 성공")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError("인코딩을 판별하지 못했습니다.")


df = read_korean_csv(SRC)
df["합계(TEU)"] = df[["수출(TEU)", "수입(TEU)", "환적(TEU)"]].sum(axis=1)
pivot = df.pivot_table(index="연도", columns="항만", values="합계(TEU)", aggfunc="sum")
pivot.to_csv(OUT / "yearly_teu_pivot.csv", encoding="utf-8-sig")
(pivot / 10000).round(1)  # 만 TEU 단위

## 6. [예제 4] 수집 파이프라인 — 수집 → 정제 → SQLite 적재 (증분·중복방지)

실무에서는 매일/매월 같은 API를 반복 호출합니다. 이때 필요한 것:
- 같은 데이터를 두 번 넣지 않기 (`PRIMARY KEY` + `INSERT OR REPLACE`)
- 언제 무엇을 수집했는지 기록 (수집 로그 테이블)
- 원본(raw)과 정제본(clean)을 분리 보관

결과 : `output/port.db` 생성

> 정기 실행이 필요하면 노트북보다 `.py` 스크립트로 내보내 cron 등에 등록하는 것이 좋습니다.
> (예: 매일 06:10 실행 `10 6 * * *  cd /path/port_data_lab && python ex04_pipeline.py`)

In [ ]:
CONTAINER_URL = "https://api.odcloud.kr/api/15147941/v1/uddi:REPLACE-WITH-YOUR-UDDI"
VESSEL_URL = "https://apis.data.go.kr/B551xxx/REPLACE_SERVICE/REPLACE_OPERATION"

DB = OUT / "port.db"

SCHEMA = """
CREATE TABLE IF NOT EXISTS container_monthly (
    yyyymm TEXT, terminal TEXT, io_type TEXT,
    full_teu INTEGER, empty_teu INTEGER, total_teu INTEGER,
    PRIMARY KEY (yyyymm, terminal, io_type)
);
CREATE TABLE IF NOT EXISTS vessel_call (
    call_sign TEXT, eta TEXT, vessel_name TEXT, vessel_kind TEXT, flag TEXT,
    gt INTEGER, berth TEXT, etd TEXT, prev_port TEXT, stay_hours REAL,
    PRIMARY KEY (call_sign, eta)
);
CREATE TABLE IF NOT EXISTS raw_payload (
    source TEXT, collected_at TEXT, payload TEXT
);
CREATE TABLE IF NOT EXISTS collect_log (
    source TEXT, collected_at TEXT, n_rows INTEGER, status TEXT, message TEXT
);
"""

### 6-1. 정제 함수 — 응답 항목명이 다르면 여기서 컬럼 매핑만 수정

In [ ]:
def clean_container(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records)
    df = df.rename(columns={"부두명": "terminal", "수출입구분": "io_type",
                            "적컨TEU": "full_teu", "공컨TEU": "empty_teu"})
    df["yyyymm"] = df["연도"].astype(int).astype(str) + df["월"].astype(int).astype(str).str.zfill(2)
    for c in ("full_teu", "empty_teu"):
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    df["total_teu"] = df["full_teu"] + df["empty_teu"]
    return df[["yyyymm", "terminal", "io_type", "full_teu", "empty_teu", "total_teu"]]


def clean_vessel(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records).rename(columns={
        "callSign": "call_sign", "vsslNm": "vessel_name", "vsslKind": "vessel_kind",
        "grtg": "gt", "berthNm": "berth", "etryDt": "eta", "tkoffDt": "etd", "prvsPort": "prev_port"})
    eta = pd.to_datetime(df["eta"], format="%Y%m%d%H%M", errors="coerce")
    etd = pd.to_datetime(df["etd"], format="%Y%m%d%H%M", errors="coerce")
    df["stay_hours"] = ((etd - eta).dt.total_seconds() / 3600).round(1)
    df["eta"], df["etd"] = eta.dt.strftime("%Y-%m-%d %H:%M"), etd.dt.strftime("%Y-%m-%d %H:%M")
    df["gt"] = pd.to_numeric(df["gt"], errors="coerce")
    df = df.dropna(subset=["call_sign", "eta"]).drop_duplicates(["call_sign", "eta"])
    return df[["call_sign", "eta", "vessel_name", "vessel_kind", "flag", "gt",
               "berth", "etd", "prev_port", "stay_hours"]]

### 6-2. 적재(UPSERT)와 작업 실행 — 한 작업이 실패해도 다른 작업은 계속

In [ ]:
def upsert(con: sqlite3.Connection, table: str, df: pd.DataFrame) -> int:
    cols = ",".join(df.columns)
    marks = ",".join("?" * len(df.columns))
    con.executemany(f"INSERT OR REPLACE INTO {table} ({cols}) VALUES ({marks})",
                    df.itertuples(index=False, name=None))
    return len(df)


def run_job(con, client, source, fetch, clean, table):
    now = datetime.now().isoformat(timespec="seconds")
    try:
        records = list(fetch())
        con.execute("INSERT INTO raw_payload VALUES (?,?,?)",
                    (source, now, json.dumps(records, ensure_ascii=False)))
        n = upsert(con, table, clean(records))
        con.execute("INSERT INTO collect_log VALUES (?,?,?,?,?)", (source, now, n, "OK", ""))
        print(f"[{source}] {n}건 적재")
    except Exception as e:  # 한 작업이 실패해도 다른 작업은 계속
        con.execute("INSERT INTO collect_log VALUES (?,?,?,?,?)", (source, now, 0, "FAIL", str(e)))
        print(f"[{source}] 실패: {e}")
    con.commit()

In [ ]:
print(f"MOCK 모드: {client.mock}")
with sqlite3.connect(DB) as con:
    con.executescript(SCHEMA)
    run_job(con, client, "container_monthly",
            lambda: client.iter_odcloud(CONTAINER_URL, per_page=500),
            clean_container, "container_monthly")
    run_job(con, client, "vessel_call",
            lambda: client.iter_xml(VESSEL_URL, num_rows=100),
            clean_vessel, "vessel_call")

    print("\n테이블별 행 수:")
    for t in ("container_monthly", "vessel_call", "collect_log"):
        print(f"  {t}: {con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:,}")

In [ ]:
# 수집 로그 확인 — 셀을 다시 실행해도 데이터 테이블은 중복 없이 유지되고 로그만 쌓인다
with sqlite3.connect(DB) as con:
    log = pd.read_sql("SELECT * FROM collect_log ORDER BY collected_at DESC", con)
log.head(10)

## 7. [예제 5] 적재된 데이터로 기본 분석·시각화

1. 연도별·부두별 컨테이너 처리량 추이
2. 월별 계절성 (전체 TEU)
3. 선종별 평균 재항시간

그림은 노트북에 표시되고 `output/fig*.png` 로도 저장됩니다.

In [ ]:
def set_korean_font():
    """설치된 한글 폰트를 찾아 지정 (Windows: Malgun Gothic, macOS: AppleGothic, Linux: Nanum/Noto CJK)."""
    candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False


set_korean_font()
with sqlite3.connect(DB) as con:
    cm = pd.read_sql("SELECT * FROM container_monthly", con)
    vc = pd.read_sql("SELECT * FROM vessel_call", con)

cm["year"] = cm["yyyymm"].str[:4]
cm["month"] = cm["yyyymm"].str[4:].astype(int)
cm.head()

In [ ]:
# 1) 연도별·부두별
yearly = cm.pivot_table(index="year", columns="terminal", values="total_teu", aggfunc="sum") / 10000
ax = yearly.plot(marker="o", figsize=(9, 5))
ax.set_title("연도별 부두별 컨테이너 처리량 (가상 데이터)")
ax.set_ylabel("만 TEU")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "fig1_yearly_terminal.png", dpi=150); plt.show()

In [ ]:
# 2) 월별 계절성
monthly = cm.groupby("month")["total_teu"].mean() / 10000
ax = monthly.plot(kind="bar", color="#2b6cb0", figsize=(9, 4))
ax.set_title("월평균 처리량 — 계절성 (가상 데이터)")
ax.set_ylabel("만 TEU"); ax.set_xlabel("월")
plt.tight_layout(); plt.savefig(OUT / "fig2_monthly_season.png", dpi=150); plt.show()

In [ ]:
# 3) 선종별 평균 재항시간
stay = vc.groupby("vessel_kind")["stay_hours"].agg(["count", "mean"]).sort_values("mean")
print("선종별 재항시간(시간):\n", stay.round(1))
ax = stay["mean"].plot(kind="barh", color="#dd6b20", figsize=(8, 4))
ax.set_title("선종별 평균 재항시간 (가상 데이터)"); ax.set_xlabel("시간")
plt.tight_layout(); plt.savefig(OUT / "fig3_stay_by_kind.png", dpi=150); plt.show()

In [ ]:
# 요약 지표
growth = yearly.sum(axis=1).pct_change().mul(100).round(1)
print("전체 처리량 전년 대비 증감률(%):\n", growth.dropna().to_string())
print("\n그림 저장:", sorted(p.name for p in OUT.glob("fig*.png")))

## 부록. 자주 만나는 오류

| 코드 | 의미 | 조치 |
|---|---|---|
| 30 | 등록되지 않은 서비스키 | Decoding 키 사용, 발급 직후면 1~2시간 대기 |
| 20 | 접근 거부 | 해당 API 활용신청 여부 확인 |
| 22 | 일일 호출량 초과 | 개발계정 한도(보통 일 10,000건) 확인 |
| 03 | 데이터 없음 | 조회 조건(기간·항코드) 확인 |
| 한글 깨짐 | CSV 인코딩 | `encoding="cp949"` / 저장 시 `utf-8-sig` |

### 항만 물류 관련 대표 데이터 (포털 검색 키워드)
- 부산항만공사: 외내항컨테이너통합집계정보, 외내항선박통합집계실시간, 외내항화물통합집계정보, 부산항 컨테이너 수송통계, 선박 입출항 집계 정보
- 인천항만공사: 항만운영 통계정보 조회, 공컨테이너 화물 통계정보 조회, 선석 이용 정보 조회, 선박 제원 정보 조회, 갑문 정보 조회
- 여수광양항만공사: 컨테이너 입출항 통계, 항만시설사용 정보, 광양항 컨테이너선 기항 현황
- 울산항만공사: 선석배정현황
- 해양수산부: 해운항만물류정보시스템(PORT-MIS) 관련 코드·통계, 수출입 물류 플랫폼 선박 입항신고 정보

데이터명·제공 여부는 수시로 바뀌므로 포털에서 최신 정보를 확인하세요.